# ATLID aerosol optical thickness (ATL_ALD_2A) — Antarctica / Ross Sea

Produces **`aot_resampled_an.nc`**: 1-second-resampled aerosol optical
thickness at 355 nm over the Ross Sea, ready for `merger_AN.ipynb`.

**Region:** 80-60 deg S, 160 deg E - 140 deg W (= 160-220 in the 0-360 convention),
polar orbit **frame `G`** (this mirrors the "#Poles" search already present in
`atl_ald_2a.ipynb`).

Requires a MAAP bearer token in `token.txt` (or paste it into `_TOKEN`).

In [ ]:
from pystac_client import Client
import fsspec
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import numpy as np
import requests
from IPython.display import Image, display
import os
import pathlib

In [ ]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

In [ ]:
EC_COLLECTION = ['EarthCAREL2Validated_MAAP']

In [ ]:
# Region of interest — Ross Sea (Antarctica). See markdown above.
LAT_MIN, LAT_MAX = -80, -60
LON360_MIN, LON360_MAX = 160, 220   # 160 deg E ... 140 deg W in the 0-360 convention
FRAME = 'G'                          # EarthCARE polar orbit frame

## Search the catalog — polar frame `G`

In [ ]:
search = catalog.search(
    collections=EC_COLLECTION,
    filter=f"productType = 'ATL_ALD_2A' and frame = '{FRAME}'",
    method='GET',
    max_items=1000,
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

In [ ]:
data = search.item_collection_as_dict()

df = pd.json_normalize(data, record_path=['features'])[
    [
        "id",
        "properties.product:type",
        "properties.updated",
        "assets.product.href",
        "assets.quicklook.href",
        "assets.enclosure_1.href",
        "assets.enclosure_2.href",
    ]
]

df.rename(columns={
    'properties.product:type': 'product_type',
    'properties.updated': 'last_modified',
    'assets.product.href': 'Zipped Product',
    'assets.quicklook.href': 'quicklook_url',
    'assets.enclosure_1.href': 'h5_url',
    'assets.enclosure_2.href': 'HDR_url',
}, inplace=True)

df.sort_values(by='id', ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)
df

In [ ]:
_TOKEN = ""
if pathlib.Path("token.txt").exists():
    with open("token.txt", "rt") as f:
        token = f.read().strip().replace("\n", "")
else:
    token = _TOKEN

fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {token}"})
ds_url = df['h5_url']

## Download and concatenate the AOT track

In [ ]:
dslist = []
for url in tqdm(ds_url.to_list()):
    f = fs.open(url, "rb")
    ds = xr.open_dataset(f, engine="h5netcdf", group="ScienceData")
    aot = ds.aerosol_optical_thickness_355nm
    aot = aot.assign_coords({"along_track": ds.time, "lat": ds.latitude, "lon": ds.longitude})
    aot = aot.rename({"along_track": "time"})
    ds.close()
    dslist.append(aot)

aot_all = xr.concat(dslist, dim="time")
aot_all

## Restrict to the Ross Sea

Frame `G` granules cover the whole polar band, so we keep only the
observations inside the Ross Sea box. Longitudes are converted to the
0-360 convention so the antimeridian crossing is a single contiguous
interval (160-220). The very same filter is re-applied in
`kmeans_AN.ipynb`, so this stays consistent end-to-end.

In [ ]:
lon360 = aot_all.lon % 360
aot_all = aot_all.where(
    (aot_all.lat > LAT_MIN) & (aot_all.lat < LAT_MAX) &
    (lon360 > LON360_MIN) & (lon360 < LON360_MAX),
    drop=True,
)

resampled_aot = aot_all.resample(time='1s').mean().dropna(dim='time', how='all')
resampled_aot

## Save

In [ ]:
resampled_aot.to_netcdf("aot_resampled_an.nc")

In [ ]:
%cp "aot_resampled_an.nc" "/home/jovyan/my-private-bucket/."